# Datashader Browser: NYC Taxi benchmark on Kaggle

This notebook benchmarks the published `datashader-browser` package in Kaggle's native Python runtime. It is a server/Python baseline, **not** a browser-WebAssembly benchmark.

With Kaggle Internet enabled, the notebook downloads the public `nyc_taxi_wide.parq` file automatically when no input dataset is attached. Attaching a Kaggle Dataset with that file is optional and avoids a repeated 385 MB download.

In [ ]:
import sys

!{sys.executable} -m pip install -q --upgrade "datashader-browser[data]==0.1.0a2"

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen

DATA_FILENAME = 'nyc_taxi_wide.parq'
DATA_URL = 'https://datasets.holoviz.org/nyc_taxi/v1/nyc_taxi_wide.parq'
DOWNLOAD_PATH = Path('/kaggle/working') / DATA_FILENAME
candidates = list(Path('/kaggle/input').rglob(DATA_FILENAME))

if candidates:
    DATA_PATH = candidates[0]
    print(f'Using attached Kaggle Dataset: {DATA_PATH}')
else:
    DATA_PATH = DOWNLOAD_PATH
    if not DATA_PATH.exists():
        print(f'Downloading {DATA_URL} to {DATA_PATH} ...')
        part_path = DATA_PATH.with_suffix(DATA_PATH.suffix + '.part')
        request = Request(DATA_URL, headers={'User-Agent': 'datashader-browser-kaggle/0.1'})
        with urlopen(request) as response, part_path.open('wb') as output:
            total = int(response.headers.get('Content-Length', 0))
            downloaded = 0
            while block := response.read(1024 * 1024):
                output.write(block)
                downloaded += len(block)
                if downloaded % (25 * 1024 * 1024) < len(block):
                    print(f'Downloaded {downloaded / 1_000_000:.0f} MB' + (f' / {total / 1_000_000:.0f} MB' if total else ''))
        part_path.replace(DATA_PATH)

print(f'Using: {DATA_PATH}')
print(f'Size: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

In [ ]:
import json
import resource
import time

import datashader.transfer_functions as tf
import pyarrow.parquet as pq
from datashader_browser import Progress, StreamingCanvas, __version__, iter_parquet_batches

# Published Web Mercator bounds. Explicit bounds preserve a one-pass pipeline.
X_RANGE = (-8_256_000, -8_209_000)
Y_RANGE = (4_964_000, 4_990_000)
BATCH_SIZE = 250_000

parquet = pq.ParquetFile(DATA_PATH)
expected_rows = parquet.metadata.num_rows
print(f'Package: {__version__}; rows: {expected_rows:,}; batch size: {BATCH_SIZE:,}')

def report(progress: Progress) -> None:
    if progress.chunks % 10 == 0 or progress.rows == expected_rows:
        print(f'aggregated {progress.rows:,}/{expected_rows:,} rows')

started = time.perf_counter()
canvas = StreamingCanvas(
    plot_width=1_200,
    plot_height=700,
    x_range=X_RANGE,
    y_range=Y_RANGE,
)
aggregate = canvas.points(
    iter_parquet_batches(
        DATA_PATH,
        batch_size=BATCH_SIZE,
        columns=['pickup_x', 'pickup_y'],
        use_threads=True,
    ),
    'pickup_x',
    'pickup_y',
    on_progress=report,
)

result = {
    'package_version': __version__,
    'rows': aggregate.attrs['source_rows'],
    'chunks': aggregate.attrs['source_chunks'],
    'batch_size': BATCH_SIZE,
    'elapsed_seconds': round(time.perf_counter() - started, 3),
    'peak_rss_mib': round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
}
assert result['rows'] == expected_rows, result
print(json.dumps(result, indent=2))

In [ ]:
image = tf.set_background(
    tf.shade(
        aggregate,
        cmap=['#111827', '#1d4ed8', '#06b6d4', '#facc15', '#fff7ed'],
        how='eq_hist',
    ),
    '#050814',
)
output_path = Path('/kaggle/working/nyc-taxi-pickups.png')
image.to_pil().save(output_path)
print(f'Plot written to: {output_path}')
image